# Titanic EDA

Exploratory data analysis for the survival-classification pipeline. The goal is to understand the data, justify the preprocessing and feature-engineering choices made in `src/preprocessing.py`, and surface caveats relevant to modelling.

> The modelling code only ever uses the Kaggle **train.csv** (it is the only file with the `Survived` label). The Kaggle **test.csv** is unlabelled and is *not* used for evaluation.


## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Make the project importable when running from notebooks/.
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data import load_labelled_data
from src.preprocessing import engineer_features

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## 2. Load the data

`load_labelled_data` uses the real Kaggle `train.csv` if present in `data/raw/`, otherwise it falls back to the bundled synthetic sample so this notebook always runs. Pass `use_sample=True` to force the sample.

In [ ]:
df = load_labelled_data(use_sample=False, download=False)
print('Shape:', df.shape)
df.head()

## 3. Schema, dtypes, and missingness

In [ ]:
df.info()

In [ ]:
# Missing-value counts and percentages.
missing = df.isna().sum().to_frame('missing')
missing['pct'] = (missing['missing'] / len(df) * 100).round(1)
missing.sort_values('missing', ascending=False)

**What to look for.** In the real Titanic data, `Age` (~20%), `Cabin` (~77%), and `Embarked` (2 rows) are missing. This motivates median imputation for `Age`, deriving a coarse `Deck` from `Cabin` (with an explicit `Unknown` level) rather than dropping it, and most-frequent imputation for `Embarked`.

## 4. Target balance

In [ ]:
ax = df[config.TARGET].value_counts().sort_index().plot(kind='bar')
ax.set_xticklabels(config.CLASS_NAMES, rotation=0)
ax.set_title('Survival counts'); ax.set_ylabel('Passengers')
plt.show()
print('Survival rate: {:.1%}'.format(df[config.TARGET].mean()))

The classes are imbalanced (~38% survived in the real data). This is why we report precision/recall/F1/ROC-AUC in addition to accuracy, and why the train/val/test splits are **stratified** on the target.

## 5. Numeric feature distributions

In [ ]:
num_cols = ['Age', 'Fare', 'SibSp', 'Parch']
df[num_cols].describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

`Fare` is strongly right-skewed and `Age` is roughly bell-shaped. We `StandardScaler` numeric features so the MLP trains stably; tree models would not need this, but a neural net benefits from standardised inputs.

## 6. Survival by categorical features

In [ ]:
cat_cols = ['Sex', 'Pclass', 'Embarked']
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, cat_cols):
    sns.barplot(data=df, x=col, y=config.TARGET, ax=ax, errorbar=None)
    ax.set_title(f'Survival rate by {col}'); ax.set_ylabel('P(survived)')
plt.tight_layout(); plt.show()

**Key signal.** Sex is the single most predictive feature (women survived far more often), followed by passenger class. This is consistent with "women and children first" and is exactly the structure the model should capture.

## 7. Engineered features

We reuse the *exact* feature-engineering function the training script uses, so the EDA reflects what the model actually sees.

In [ ]:
feat = engineer_features(df)
feat['Survived'] = df[config.TARGET].values
feat.head()

In [ ]:
# FamilySize / IsAlone and Title vs. survival.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.barplot(data=feat, x='FamilySize', y='Survived', ax=axes[0], errorbar=None)
axes[0].set_title('Survival by family size')
sns.barplot(data=feat, x='IsAlone', y='Survived', ax=axes[1], errorbar=None)
axes[1].set_title('Survival: alone vs. not')
sns.barplot(data=feat, x='Title', y='Survived', ax=axes[2], errorbar=None)
axes[2].set_title('Survival by title'); axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

Mid-size families fare better than singletons or very large families, and `Title` (Mr/Mrs/Miss/Master/...) encodes sex + age + social status in one feature - which is why we engineer it from `Name`.

## 8. Correlations (numeric view)

In [ ]:
corr = feat.drop(columns=['Title', 'Deck', 'Sex', 'Embarked'], errors='ignore')\
           .apply(pd.to_numeric, errors='coerce').corr()
plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix (numeric & engineered)'); plt.show()

## 9. EDA conclusions -> design choices

- **Impute** `Age`/`Fare` with the median (skewed) and categoricals with the mode; keep `Cabin` as a coarse `Deck` with an explicit `Unknown` level.
- **Scale** numeric features (the MLP needs standardised inputs).
- **One-hot encode** categoricals with `handle_unknown='ignore'` for robust inference.
- **Engineer** `FamilySize`, `IsAlone`, `Title`, `Deck` - all row-wise and therefore leakage-free.
- **Stratify** splits on `Survived` because the target is imbalanced.
- **Report** precision/recall/F1/ROC-AUC, not just accuracy.
- **Avoid leakage**: the preprocessor is fit on the training split only (see `train.py`).
